# 재확인 마무리 — 영상 2개, 질문 2개

재확인(`adjudicate.ipynb`)에서 메모만 남긴 항목은 메모대로 결정을 채웠다. 다음 두 곳은 **영상을 직접 보고 판단해야 하는 것**이라 여기서 묻는다.
각 셀을 실행하면 영상이 나오고, 아래 입력칸에 번호를 넣으면 `adjudication.csv` 에 저장된다.

왼쪽 위 노란 숫자 = 녹화 시각, **노란 테두리·노란 막대 = 주목할 구간**. 영상 아래 버튼으로 속도를 바꾸고, 막대를 끌어 원하는 시각으로 간다.

In [ ]:
import base64
import os
import sys
import uuid
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

ROOT = Path(os.environ.get("CRUSHING_ROOT") or next(q for q in [Path.cwd(), *Path.cwd().parents] if (q / "analysis" / "core.py").exists()))   # project folder (contains analysis/)
sys.path.insert(0, str(ROOT / "analysis"))
import review_clips as rc  # noqa: E402

OUTF = ROOT / "outputs" / "continuous_annotation" / "adjudication.csv"


def show(rec, pen, t0, t1, h0, h1, width=600):
    vid = "v" + uuid.uuid4().hex[:8]
    b64 = base64.b64encode(Path(rc.make_range_clip(rec, pen, t0, t1, h0, h1)).read_bytes()).decode()
    btn = "".join(f'<button style="margin-right:6px" onclick="document.getElementById(\'{vid}\').playbackRate={r}">{r}×</button>'
                  for r in (1, 2, 4, 8))
    display(HTML(f'<video id="{vid}" width="{width}" controls muted playsinline><source src="data:video/mp4;base64,{b64}" '
                 f'type="video/mp4"></video><div>재생 속도: {btn}</div>'))


def save(items, choice, table):
    a = pd.read_csv(OUTF)
    for it in items:
        k = a.index[a["item"] == it][0]
        for col, val in dict(table[choice], decision_source="follow-up question with video",
                             decided_at=pd.Timestamp.now().isoformat(timespec="seconds")).items():
            a[col] = a[col].astype(object)
            a.at[k, col] = val
    a.to_csv(OUTF, index=False)
    print("저장:", items, "->", table[choice]["decision"])


def ask(prompt, table):
    while True:
        c = input(prompt).strip()
        if c in table:
            return c
        print("번호를 확인하세요:", ", ".join(table))

## 질문 1 — 13번 (A_kt1959 펜 R)
연속 주석에서 **8:26 에 깔림**을 표시했고 끝(`e`)은 없다. 재확인 때 **15:40** 경보 장면에 "영상 처음부터 깔려 있음"이라고 메모했다.

아래 영상은 **8:16 – 15:50** 이다(노란 구간 = 8:26 표시부터 15:40 경보까지). 4–8배속으로 보면서,
**8:26 에 깔린 새끼가 15:40 까지 계속 모돈 밑에 있었는지** 본다.

| 입력 | 뜻 | 처리 |
|---|---|---|
| `1` | 8:26 의 그 새끼가 15:40 까지 계속 깔려 있었다 | 15:40 경보 = 같은 사건의 중복 경보 |
| `2` | 중간에 빠져나왔다가 다시 깔렸거나, 다른 새끼다(언제 깔렸는지 모름) | 15:40 경보는 판단에서 제외 |
| `3` | 모르겠다 | 판단에서 제외 |

In [ ]:
show("A_kt1959", "R", 496.0, 950.0, 506.0, 940.3)
Q1 = {"1": dict(decision="continuation", ref_type="trapping", ref_onset_s=506.0, new_type=""),
      "2": dict(decision="unclear", ref_type="", ref_onset_s=float("nan"), new_type=""),
      "3": dict(decision="unclear", ref_type="", ref_onset_s=float("nan"), new_type="")}
save([13], ask("13번: 1 계속 깔려 있었음 | 2 다른 깔림/언제 깔렸는지 모름 | 3 모르겠음 > ", Q1), Q1)

## 질문 2 — 15·16번 (A_kt2044 펜 R)
연속 주석에서 **0:35 에 깔렸다가 1:25 에 빠져나온 새끼**를 표시했다. 재확인 때 **2:52** 와 **5:08–5:33** 장면에
"죽어 있는 것 같은 자돈이 보임"이라고 메모했다.

아래 영상은 **0:25 – 5:45** 이다(노란 구간 = 0:35 깔림 ~ 1:25 빠져나옴). **1:25 에 빠져나온 새끼를 눈으로 따라가서**,
2:52 와 5:08 에 보이는 죽은 것 같은 새끼가 그 새끼인지 본다.

| 입력 | 뜻 | 처리 |
|---|---|---|
| `1` | 1:25 에 빠져나온 그 새끼가 죽은 것 같다 | 0:35 사건을 crushing 으로 바꾸고, 2:52·5:08 경보 = 그 사건의 중복 경보 |
| `2` | 원래부터 죽어 있던 **다른** 새끼다 | 2:52·5:08 경보는 판단에서 제외 |
| `3` | 모르겠다 | 판단에서 제외 |

In [ ]:
show("A_kt2044", "R", 25.0, 345.0, 35.0, 85.0)
Q2 = {"1": dict(decision="continuation", ref_type="trapping", ref_onset_s=35.0, new_type="crushing"),
      "2": dict(decision="preexisting_dead", ref_type="", ref_onset_s=float("nan"), new_type=""),
      "3": dict(decision="unclear", ref_type="", ref_onset_s=float("nan"), new_type="")}
save([15, 16], ask("15·16번: 1 빠져나온 그 새끼가 죽음 | 2 원래 죽어 있던 다른 새끼 | 3 모르겠음 > ", Q2), Q2)